# Actividad 4: Formatos de datos (año 2009)

Vamos a coger el fichero `2009.csv` de retrasos de vuelos y guardarlo en tres formatos distintos: **Parquet**, **ORC** y **Avro**. Después compararemos cuánto ocupa cada uno y cuánto tarda en crearse.

## 1. Los datos que usamos

- **Fuente:** son datos **secundarios** y **externos**. No los hemos recogido nosotros: vienen de un dataset público de Kaggle.
- **¿Sirven para el problema?** Sí. Queremos estudiar retrasos, y el fichero tiene fechas (`FL_DATE`), así que son datos de **series temporales**.
- **Estructura:** son datos **estructurados**, organizados en filas (vuelos) y columnas (campos).

In [ ]:
!pip install -q fastavro

In [ ]:
import os
import time
import pandas as pd
from fastavro import writer, parse_schema

## 2. Leer el CSV

In [ ]:
# Busca 2009.csv dentro de los datos añadidos al notebook
for carpeta, _, ficheros in os.walk('/kaggle/input'):
    if '2009.csv' in ficheros:
        ruta_csv = os.path.join(carpeta, '2009.csv')
print(ruta_csv)

df = pd.read_csv(ruta_csv)
df.head()

## 3. Tipos de datos de las columnas que usaremos

| Columna | Qué es | Tipo de dato |
|---|---|---|
| `FL_DATE` | Fecha del vuelo | Fecha (para series temporales) |
| `OP_CARRIER` | Código de la aerolínea (AA, DL...) | **Cualitativo nominal**: una categoría sin orden |
| `DEP_DELAY` | Minutos de retraso en la salida | **Cuantitativo continuo**: un número medido |

## 4. Transformar los datos

Según la lectura, transformar es cambiar el formato, la estructura o los valores de los datos. Aquí hacemos dos transformaciones:
1. **Eliminar un campo:** el CSV trae una columna vacía al final (`Unnamed: 27`) que no sirve.
2. **Guardar en otro formato:** pasamos de CSV a Parquet, ORC y Avro.

In [ ]:
df = df.drop(columns=['Unnamed: 27'])
print(df.shape)

## 5. Guardar en Parquet

Formato **columnar** (guarda los datos columna a columna) y comprimido. Es muy usado en Big Data.

In [ ]:
inicio = time.time()
df.to_parquet('air2009.parquet')
tiempo_parquet = time.time() - inicio
print(f'Parquet creado en {tiempo_parquet:.2f} segundos')

## 6. Guardar en ORC

También es **columnar** y comprimido. Se creó para el ecosistema Hadoop/Hive.

In [ ]:
inicio = time.time()
df.to_orc('air2009.orc')
tiempo_orc = time.time() - inicio
print(f'ORC creado en {tiempo_orc:.2f} segundos')

## 7. Guardar en Avro (solo 3 columnas)

Avro es un formato **por filas** que guarda junto a los datos un **esquema**: la descripción de cada campo y su tipo. Solo guardamos la fecha, la aerolínea y el retraso. `DEP_DELAY` puede estar vacío (vuelos cancelados), por eso su tipo es `null` o `double`.

In [ ]:
esquema = parse_schema({
    'type': 'record',
    'name': 'Vuelo',
    'fields': [
        {'name': 'FL_DATE',    'type': 'string'},
        {'name': 'OP_CARRIER', 'type': 'string'},
        {'name': 'DEP_DELAY',  'type': ['null', 'double']}
    ]
})

# Nos quedamos con las 3 columnas y cambiamos los vacíos (NaN) por None
pequeno = df[['FL_DATE', 'OP_CARRIER', 'DEP_DELAY']]
pequeno = pequeno.astype(object).where(pequeno.notna(), None)
registros = pequeno.to_dict('records')

inicio = time.time()
with open('air2009_small.avro', 'wb') as f:
    writer(f, esquema, registros)
tiempo_avro = time.time() - inicio
print(f'Avro creado en {tiempo_avro:.2f} segundos')

## 8. Comparar tamaños y tiempos

In [ ]:
def mb(ruta):
    return round(os.path.getsize(ruta) / 1024 / 1024, 2)

resumen = pd.DataFrame({
    'Fichero':     ['2009.csv (original)', 'air2009.parquet', 'air2009.orc', 'air2009_small.avro'],
    'Tamaño (MB)': [mb(ruta_csv), mb('air2009.parquet'), mb('air2009.orc'), mb('air2009_small.avro')],
    'Tiempo (s)':  ['-', round(tiempo_parquet, 2), round(tiempo_orc, 2), round(tiempo_avro, 2)]
})
resumen

## 9. Conclusión

Parquet y ORC ocupan mucho menos que el CSV porque guardan los datos por columnas y los comprimen. El Avro es pequeño porque solo tiene 3 columnas, aunque tarda más en crearse porque escribe los vuelos fila a fila.